# Semantic cache: the linear algebra, step by step

This notebook walks through every matrix operation behind the cache, using the project's own code in `semcache/linalg.py` (written from scratch, checked against NumPy).

1. TF-IDF by hand on three questions
2. The TF-IDF matrix of the FAQ and its truncated SVD
3. Checking our SVD against NumPy
4. Folding a new query into the latent space
5. The Gram matrix of cached questions: TF-IDF vs LSA
6. The cache in action
7. Interactive: the angle picture with a threshold slider
8. Locality-sensitive hashing: theory vs experiment

**Running in Google Colab:** upload the project folder (or `git clone` your team's repo), then run:

```python
%cd semantic-cache
%pip install -r requirements.txt
```

In [ ]:
import os, sys
from pathlib import Path

# Work from the project root whether the notebook is opened from notebooks/ or the root.
ROOT = Path.cwd() if (Path.cwd() / "semcache").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import HTML, Image, display

from semcache import linalg as la
from semcache.data import faq_corpus, load_faq
from semcache.embedder import LSAEmbedder
from semcache.text import analyze

np.set_printoptions(precision=3, suppress=True)
print("project root:", ROOT)

## 1. TF-IDF by hand

For a term $t$ in document $d$, with $N$ documents and $\mathrm{df}(t)$ of them containing $t$:

$$
\mathrm{tfidf}(t, d) = \mathrm{tf}(t, d)\,\cdot\,\ln\frac{N}{\mathrm{df}(t)}
$$

A word that appears in every document gets $\ln 1 = 0$ and disappears. Here are three short queries.

In [ ]:
docs = [["cheap", "flights", "to", "goa"],
        ["cheap", "train", "to", "mumbai"],
        ["flights", "to", "mumbai"]]
vocab = ["cheap", "flights", "to", "goa", "train", "mumbai"]
index = {t: i for i, t in enumerate(vocab)}

TF = la.term_frequency_matrix([[index[w] for w in d] for d in docs], len(vocab))
df = la.document_frequency(TF)
idf = la.inverse_document_frequency(df, n_docs=3, smooth=False)
X_raw = la.tfidf_matrix(TF, idf, normalize=False)

display(pd.DataFrame({"df": df, "idf = ln(N/df)": idf}, index=vocab).round(3))
display(pd.DataFrame(X_raw.toarray(), columns=vocab, index=["d1", "d2", "d3"]).round(3))

Each row is then scaled to unit length, $\hat x = x / \lVert x \rVert_2$, so the dot product of two rows *is* their cosine similarity:

$$
\cos\theta = \frac{a \cdot b}{\lVert a \rVert\,\lVert b \rVert} = \hat a \cdot \hat b
$$

In [ ]:
X = la.l2_normalize_rows(X_raw)
print("row norms:", la.row_norms(X))
print("cosine similarity matrix X X^T:")
print((X @ X.T).toarray())

## 2. The FAQ's TF-IDF matrix and its truncated SVD

The project fits on FAQ questions plus their paraphrases (and on Quora questions, once downloaded). The SVD keeps the $k$ strongest directions:

$$
X \approx U_k\, \Sigma_k\, V_k^{\top}, \qquad \Sigma_k = \mathrm{diag}(\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_k)
$$

`la.randomized_svd` does this with our own Gram-Schmidt QR and Jacobi eigen-solver: no `numpy.linalg.svd`.

In [ ]:
items = load_faq("data/faq.csv")
corpus = faq_corpus(items)
emb = LSAEmbedder().fit(corpus, verbose=True)

X_faq = emb.tfidf(corpus)
density = X_faq.nnz / (X_faq.shape[0] * X_faq.shape[1])
print(f"TF-IDF matrix: {X_faq.shape[0]} x {X_faq.shape[1]}, {X_faq.nnz} non-zeros ({density:.2%} filled)")
print("top 10 singular values:", emb.s[:10])

In [ ]:
from semcache.evaluate import plot_spectrum

NB_DIR = Path("artifacts/notebook")          # the notebook never touches your main model or cache
NB_DIR.mkdir(parents=True, exist_ok=True)
display(Image(plot_spectrum(emb, NB_DIR / "spectrum.png")))

Each column of $V_k$ is a unit vector over the vocabulary. Its largest entries show which words that direction groups together:

In [ ]:
pd.DataFrame([{"direction": d["direction"], "sigma": d["sigma"], "top terms": ", ".join(d["terms"])}
              for d in emb.direction_terms(8, 6)]).set_index("direction")

## 3. Checking our SVD against NumPy

The FAQ matrix is small enough to densify, so we can compare with LAPACK. `exact_svd` gets the singular values from the eigenvalues of the Gram matrix, $X^{\top}X = V \Sigma^2 V^{\top}$, using our Jacobi solver. `randomized_svd` is what large (Quora-sized) sparse matrices use: it is faster, at the cost of a small error that more power iterations shrink.

We also check the Eckart-Young theorem: the rank-$k$ truncation is the best rank-$k$ approximation, with error

$$
\lVert X - X_k \rVert_F = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \cdots}
$$

In [ ]:
dense = X_faq.toarray()
k = 20
s_numpy = np.linalg.svd(dense, compute_uv=False)

U_e, s_e, V_e = la.exact_svd(dense, k)
print("exact_svd      max relative error:", np.max(np.abs(s_e - s_numpy[:k]) / s_numpy[:k]))
for q in (2, 7, 15):
    _, s_r, _ = la.randomized_svd(X_faq, k, power_iters=q, seed=0)
    print(f"randomized_svd max relative error with {q:2d} power iterations:",
          np.max(np.abs(s_r - s_numpy[:k]) / s_numpy[:k]))

print("U orthonormal:", np.allclose(U_e.T @ U_e, np.eye(k)), "  V orthonormal:", np.allclose(V_e.T @ V_e, np.eye(k)))
ours = np.linalg.norm(dense - (U_e * s_e) @ V_e.T)
best = np.sqrt(np.sum(s_numpy[k:] ** 2))
print(f"rank-{k} error: ours {ours:.6f}, Eckart-Young optimum {best:.6f}")

## 4. Folding a new query into the latent space

A query $q$ (a TF-IDF row) is never part of the SVD. Its latent coordinates come from the fitted $V_k$:

$$
\hat q = q\, V_k\, \Sigma_k^{-1}
$$

For a training row this gives exactly its row of $U_k$, because $X V_k = U_k \Sigma_k$. The cache compares $\hat q\,\Sigma_k = q V_k$ (normalised) with each cached vector.

In [ ]:
U_k, s_k, V_k = la.exact_svd(dense, emb.dim)
print("fold-in of training rows equals their U rows:",
      np.allclose(la.fold_in(X_faq[:3], V_k, s_k), U_k[:3], atol=1e-8))

pairs = [("low cost airfare to goa", "How can I find cheap flights to Goa?"),
         ("my laptop is really slow", "How can I speed up a slow laptop?"),
         ("i forgot my password, how can i reset it", "How do I reset my password if I forgot it?"),
         ("who won the cricket world cup", "How can I find cheap flights to Goa?")]
rows = []
for a, b in pairs:
    tfidf_cos = la.rowwise_cosine(emb.tfidf([a]), emb.tfidf([b]))[0]
    lsa_cos = float(emb.embed_one(a) @ emb.embed_one(b))
    rows.append({"query": a, "cached question": b, "TF-IDF cos": round(tfidf_cos, 3), "LSA cos": round(lsa_cos, 3)})
pd.DataFrame(rows)

## 5. The Gram matrix: who looks similar to whom

Stack the unit vectors of all cached questions into $C$. Then $S = C\,C^{\top}$ holds every pairwise cosine at once. The questions are sorted by topic, with thin lines between topics, so look at the blocks along the diagonal: LSA adds within-topic similarity that raw TF-IDF misses, because it has learned which words occur together. On the small FAQ corpus the effect is mild; rebuild with Quora and it grows.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

order = sorted(range(len(items)), key=lambda i: items[i].topic)
questions = [items[i].question for i in order]
C_lsa = emb.embed(questions)
C_tfidf = emb.tfidf(questions)

blues = LinearSegmentedColormap.from_list(
    "blues", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), dpi=110)
for ax, M, title in ((axes[0], (C_tfidf @ C_tfidf.T).toarray(), "TF-IDF: C Cᵀ"),
                     (axes[1], C_lsa @ C_lsa.T, f"LSA (k = {emb.dim}): C Cᵀ")):
    im = ax.imshow(np.clip(M, 0, 1), cmap=blues, vmin=0, vmax=1)
    for edge in np.cumsum(pd.Series([items[i].topic for i in order]).value_counts(sort=False).values)[:-1]:
        ax.axhline(edge - 0.5, color="#898781", lw=0.6)
        ax.axvline(edge - 0.5, color="#898781", lw=0.6)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8, label="cosine similarity")
plt.show()

topics = pd.Series([items[i].topic for i in order]).value_counts(sort=False)
print("topics in order:", dict(topics))

## 6. The cache in action

`SemanticCacheService` ties it together: embed, look up, and on a miss ask the LLM (here the stub, so the notebook runs anywhere) and store the answer.

In [ ]:
from semcache.config import Settings
from semcache.pipeline import SemanticCacheService, build_artifacts

os.environ["SEMCACHE_ARTIFACTS"] = str(NB_DIR)   # separate from artifacts/model and artifacts/cache
settings = Settings.load()
settings.llm.backend = "stub"                    # no Ollama needed for the notebook
build_artifacts(settings, verbose=False)         # fresh model + FAQ-warmed cache in artifacts/notebook
service = SemanticCacheService.from_settings(settings, verbose=False)

log = []
for q in ["i forgot my password, how can i reset it", "who won the cricket world cup",
          "Who won the cricket world cup?", "my laptop is really slow"]:
    r = service.ask(q)
    log.append({"query": q, "hit": r.hit, "score": r.score, "matched": r.matched_query,
                "source": r.source, "lookup ms": r.timings_ms["lookup"]})
pd.DataFrame(log)

## 7. Interactive: the angle picture

Cosine similarity is the cosine of an angle. The query points straight up; each cached question sits at $\theta = \arccos(\cos\theta)$ from it. A cached answer is reused when it falls inside the cone of half-angle $\arccos\tau$. Move the slider to see the trade-off. (In Colab the widgets work as-is; in VS Code install the Jupyter extension.)

In [ ]:
import ipywidgets as widgets
from semcache.viz import angle_fan_svg

text = widgets.Text(value="cheap flight tickets to goa", description="Question", layout=widgets.Layout(width="70%"))
tau = widgets.FloatSlider(value=settings.cache.threshold, min=0.3, max=0.99, step=0.01, description="τ")
out = widgets.Output()

def redraw(*_):
    with out:
        out.clear_output(wait=True)
        if not text.value.strip():
            return
        result, _ = service.lookup(text.value, threshold=tau.value, record=False)
        matches = [m.to_dict() for m in result.matches]
        display(HTML(angle_fan_svg(matches, tau.value, result.method == "exact")))
        display(pd.DataFrame([{"rank": f"c{i + 1}", "cached question": m["query"], "cos": m["score"],
                               "angle": round(np.degrees(np.arccos(min(1, max(0, m["score"])))), 1),
                               "hit": m["score"] >= tau.value} for i, m in enumerate(matches)]))

text.observe(redraw, names="value")
tau.observe(redraw, names="value")
display(widgets.VBox([text, tau, out]))
redraw()

## 8. Locality-sensitive hashing

With random hyperplane normals $r_1, \dots, r_b$, a vector hashes to the sign pattern $[\,r_1\cdot x > 0, \dots, r_b \cdot x > 0\,]$. Two vectors at angle $\theta$ agree on one bit with probability $1 - \theta/\pi$, so with $b$ bits per table and $L$ tables a neighbour becomes a candidate with probability

$$
P(\text{candidate}) = 1 - \left(1 - \left(1 - \frac{\theta}{\pi}\right)^{b}\right)^{L}
$$

In [ ]:
rng = np.random.default_rng(0)
dim, b, L = emb.dim, 12, 8
lsh = la.RandomHyperplaneLSH(dim, n_bits=b, n_tables=L, seed=1)
base = la.l2_normalize_rows(rng.standard_normal((400, dim)))
lsh.add(range(len(base)), base)

cos_grid = np.linspace(0.3, 1.0, 15)
empirical = []
for c in cos_grid:
    found = 0
    for i in range(len(base)):
        r = rng.standard_normal(dim)
        r = la.l2_normalize_rows(r - (r @ base[i]) * base[i])
        q = c * base[i] + np.sqrt(max(0.0, 1 - c * c)) * r   # exactly cos = c to base[i]
        found += i in lsh.candidates(q)
    empirical.append(found / len(base))

fig, ax = plt.subplots(figsize=(7, 3.8), dpi=110)
ax.plot(cos_grid, la.lsh_collision_probability(cos_grid, b, L), color="#2a78d6", lw=2, label="theory")
ax.plot(cos_grid, empirical, "o", color="#eb6834", ms=6, mec="#fcfcfb", mew=1.5, label="measured")
ax.set_xlabel("cosine similarity to the query"); ax.set_ylabel("probability of being a candidate")
ax.set_title(f"LSH with b = {b} bits and L = {L} tables", loc="left")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#e1e0d9"); ax.legend(frameon=False)
plt.show()

## Next: the real evaluation

Download Quora Question Pairs, rebuild, and evaluate from a terminal:

```bash
python scripts/download_qqp.py
python -m semcache build
python -m semcache evaluate
```

The report in `artifacts/eval/report.md` has precision, recall and F1 for every threshold, the rank sweep over $k$, the LSA vs TF-IDF blend sweep, a cache simulation and the LSH latency benchmark.

In [ ]:
report = Path("artifacts/eval/report.md")
print(report.read_text()[:1500] if report.exists() else "No report yet: run `python -m semcache evaluate`.")